# S&P 500 — Full-Covariance vs Factor-Analysis Mixtures

This notebook fits **eight** Generalized-Hyperbolic-family mixture models to
**all** S&P 500 stocks and compares full-covariance versus low-rank
factor-analysis storage of the dispersion matrix $\Sigma$.

## Distributions compared

**Full-covariance family** (stores Cholesky of $\Sigma$):

1. Variance Gamma — $Y \sim \mathrm{Gamma}(\alpha, \beta)$
2. Normal-Inverse-Gamma — $Y \sim \mathrm{InvGamma}(\alpha, \beta)$
3. Normal-Inverse-Gaussian — $Y \sim \mathrm{InvGaussian}(\mu_{IG}, \lambda)$
4. Generalized Hyperbolic — $Y \sim \mathrm{GIG}(p, a, b)$

**Factor-analysis family** (stores $\Sigma = F F^\top + \mathrm{diag}(D)$,
with $F \in \mathbb{R}^{d \times r}$, $r=50$):

5. Factor VG, 6. Factor NInvG, 7. Factor NIG, 8. Factor GH

The four GIG-subordinator parameters $(p, a, b)$ specialise as

| Family | $p$ | $a$ | $b$ |
|---|---|---|---|
| VG (Gamma) | $\alpha$ | $2\beta$ | $0$ |
| NInvG (InvGamma) | $-\alpha$ | $0$ | $2\beta$ |
| NIG (InvGaussian) | $-1/2$ | $\lambda/\mu_{IG}^2$ | $\lambda$ |
| GH (GIG) | $p$ | $a$ | $b$ |

so we report the GIG triple uniformly across all eight models.

## Methodology

- **Data**: full S&P 500 universe, daily log returns
- **Train / test split**: chronological, first half / second half
- **Backends**: user-selectable JAX vs CPU for E-step / M-step
- **Regularization**: GH and FactorGH default to `'det_sigma_x'`
  (anchor $|\Sigma|$ to the initial NIG/VG/NInvG-warmstarted scale),
  which keeps their displayed $(\gamma, \Sigma, a, b)$ on the same
  scale as the other families. See §2 for alternatives
  (`'det_sigma_one'`, `'a_eq_b'`).
- **Heatmaps**: parameters restricted to the seven "magnificent" stocks
  (AAPL, MSFT, GOOGL, AMZN, NVDA, META, TSLA) for readability
- **Metrics**: mean log-likelihood (train + test); orbit-invariant
  parameter summaries (§4.1); spectrum / condition number of $\Sigma$;
  random-portfolio KS / AD; portfolio quantile match


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import ks_2samp, anderson_ksamp
import warnings, time, os
warnings.filterwarnings('ignore')

jax.config.update("jax_enable_x64", True)

from normix import (
    VarianceGamma, NormalInverseGamma, NormalInverseGaussian, GeneralizedHyperbolic,
    FactorVarianceGamma, FactorNormalInverseGamma,
    FactorNormalInverseGaussian, FactorGeneralizedHyperbolic,
)

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print("Imports successful — JAX devices:", jax.devices())


## 1. Load S&P 500 Stock Returns


In [ ]:
DATA_PATH = '../data/sp500_returns.csv'
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"{DATA_PATH} not found. Run scripts/download_sp500_data.py first.")

log_returns = pd.read_csv(DATA_PATH, index_col='Date', parse_dates=True)
log_returns = log_returns.dropna(axis=1, how='any')

print(f"Shape: {log_returns.shape}  (T={log_returns.shape[0]} days, "
      f"d={log_returns.shape[1]} stocks)")
print(f"Date range: {log_returns.index[0].date()}  →  "
      f"{log_returns.index[-1].date()}")

selected_tickers = list(log_returns.columns)
d = len(selected_tickers)


In [ ]:
n_total = len(log_returns)
n_train = n_total // 2

train_df = log_returns.iloc[:n_train]
test_df = log_returns.iloc[n_train:]

returns_train = train_df.values
returns_test = test_df.values

X_train = jnp.asarray(returns_train, dtype=jnp.float64)
X_test = jnp.asarray(returns_test, dtype=jnp.float64)

print(f"Train: {returns_train.shape[0]:>5} days × {returns_train.shape[1]} stocks "
      f"({train_df.index[0].date()} → {train_df.index[-1].date()})")
print(f"Test : {returns_test.shape[0]:>5} days × {returns_test.shape[1]} stocks "
      f"({test_df.index[0].date()} → {test_df.index[-1].date()})")


In [ ]:
# Magnificent 7: focus stocks for parameter heatmaps.
MAGNIFICENT_7 = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA']
MAG7 = [t for t in MAGNIFICENT_7 if t in selected_tickers]
mag7_idx = np.array([selected_tickers.index(t) for t in MAG7])
print("Magnificent 7 indices:", dict(zip(MAG7, mag7_idx.tolist())))


## 2. Configuration

Edit this cell to switch backends or change the EM budget. The defaults
balance accuracy and runtime for `d ≈ 470` stocks; `'jax'` is faster on
GPU but `'cpu'` is competitive on a small machine because the E-step
Bessel work is offloaded to `scipy.special.kve`.

### Regularization choice for GH / FactorGH

GH (and its factor variant) has a one-parameter scale-equivalence orbit:
under $Y \to s\,Y$ the parameters transform as

$$
(\gamma, \Sigma) \to (\gamma/s,\; \Sigma/s),\qquad
(a, b) \to (a/s,\; b\cdot s).
$$

A regularization picks one canonical point on this orbit each EM
iteration. Three reasonable choices ship with `normix`:

- `'det_sigma_one'` — historical default; sets $|\Sigma| = 1$.
  Sensitive to the small-eigenvalue tail in high dimensions, which
  inflates the displayed $\gamma$ and `a` (the original "huge γ" issue).
- `'det_sigma_x'` — anchors $|\Sigma|$ to the **initial** model's
  determinant. Since the GH `default_init` runs short fits of NIG / VG
  / NInvG, $|\Sigma_0|$ is on the data's empirical scale, and the
  fitted $\Sigma$ stays comparable to the VG / NIG / NInvG fits.
  **Recommended default for fair parameter comparison.**
- `'a_eq_b'` — sets $a = b = \sqrt{ab}$ (subordinator-anchored;
  no-op for VG / NInvG where one of $a, b$ is zero).

VG / NInvG / NIG default to `'none'` (their EM is well-behaved without
rescaling).


In [ ]:
# ---- Per-distribution backend / solver knobs ----
E_STEP_BACKEND = 'jax'    # 'jax' | 'cpu'
M_STEP_BACKEND = 'cpu'    # 'jax' | 'cpu'
M_STEP_METHOD  = 'newton' # 'newton' | 'lbfgs' | 'bfgs'

# ---- EM convergence budget ----
MAX_ITER = 30
TOL = 1e-2

# ---- Factor-analysis rank ----
R_FACTORS = 50

# ---- GH / FactorGH regularization ---------------------------------
# 'det_sigma_x' anchors |Σ| to the initial Σ (data-scale), so γ and Σ
# stay comparable to the VG / NIG / NInvG fits. Try also:
#   'det_sigma_one' — original |Σ|=1 convention (large displayed γ)
#   'a_eq_b'        — set a = b = √(ab)
#   'none'          — let EM drift
GH_REGULARIZATION = 'det_sigma_x'

print(f"Configuration:")
print(f"  E-step backend     : {E_STEP_BACKEND}")
print(f"  M-step backend     : {M_STEP_BACKEND}")
print(f"  M-step method      : {M_STEP_METHOD}")
print(f"  max_iter / tol     : {MAX_ITER} / {TOL}")
print(f"  factor rank r      : {R_FACTORS}")
print(f"  GH regularization  : {GH_REGULARIZATION}")


## 3. Fit All Eight Distributions


In [ ]:
fitted = {}            # name -> fitted model
em_results = {}        # name -> EMResult (for diagnostics)
fit_times = {}         # name -> wall-clock seconds


def _fit(name, init, *, regularization='none'):
    print(f"\n→ Fitting {name} ({type(init).__name__})...")
    t0 = time.time()
    res = init.fit(
        returns_train, max_iter=MAX_ITER, tol=TOL, verbose=1,
        regularization=regularization,
        e_step_backend=E_STEP_BACKEND, m_step_backend=M_STEP_BACKEND,
        m_step_method=M_STEP_METHOD,
    )
    fit_times[name] = time.time() - t0
    fitted[name] = res.model
    em_results[name] = res
    train_ll = float(res.model.marginal_log_likelihood(X_train))
    test_ll  = float(res.model.marginal_log_likelihood(X_test))
    print(f"  done in {fit_times[name]:.1f}s — train LL={train_ll:.4f}  "
          f"test LL={test_ll:.4f}")


In [ ]:
_fit('VG', VarianceGamma.default_init(returns_train))


In [ ]:
_fit('NInvG', NormalInverseGamma.default_init(returns_train))


In [ ]:
_fit('NIG', NormalInverseGaussian.default_init(returns_train))


In [ ]:
_fit('GH', GeneralizedHyperbolic.default_init(returns_train),
     regularization=GH_REGULARIZATION)


In [ ]:
_fit('FactorVG',
     FactorVarianceGamma.default_init(returns_train, r=R_FACTORS))


In [ ]:
_fit('FactorNInvG',
     FactorNormalInverseGamma.default_init(returns_train, r=R_FACTORS))


In [ ]:
_fit('FactorNIG',
     FactorNormalInverseGaussian.default_init(returns_train, r=R_FACTORS))


In [ ]:
_fit('FactorGH',
     FactorGeneralizedHyperbolic.default_init(returns_train, r=R_FACTORS),
     regularization=GH_REGULARIZATION)


## 4. Fitted Parameters

We extract three per-model summaries:

- **Subordinator $(p, a, b)$** — the GIG triple, into which all four
  subordinator families embed. Closed-form-subordinator distributions
  have one or both of $a, b$ degenerate to $0$.
- **Location $\mu$ and skewness $\gamma$** — heatmap restricted to the
  Magnificent 7 stocks across the eight fitted models, so structural
  agreement is visible.
- **Dispersion $\Sigma$** — $7 \times 7$ submatrix of the Magnificent
  7 stocks for each model.


In [ ]:
def gig_pab(model, name):
    """Map any fitted subordinator to its GIG (p, a, b) representation.

    See `docs/theory/gh.rst` for the embedding of each subordinator
    family in GIG.
    """
    if name in ('VG', 'FactorVG'):
        return float(model.alpha), float(2.0 * model.beta), 0.0
    if name in ('NInvG', 'FactorNInvG'):
        return float(-model.alpha), 0.0, float(2.0 * model.beta)
    if name in ('NIG', 'FactorNIG'):
        return -0.5, float(model.lam / (model.mu_ig ** 2)), float(model.lam)
    if name in ('GH', 'FactorGH'):
        return float(model.p), float(model.a), float(model.b)
    raise KeyError(name)


def get_sigma(model):
    return np.asarray(model.sigma())


def get_mu(model):
    return np.asarray(model.mu)


def get_gamma(model):
    return np.asarray(model.gamma)


DIST_ORDER = ['VG', 'NInvG', 'NIG', 'GH',
              'FactorVG', 'FactorNInvG', 'FactorNIG', 'FactorGH']
DIST_ORDER = [d for d in DIST_ORDER if d in fitted]


In [ ]:
gig_rows = []
for name in DIST_ORDER:
    p, a, b = gig_pab(fitted[name], name)
    gig_rows.append({'Distribution': name,
                     'p': p, 'a': a, 'b': b,
                     'sqrt(ab)': np.sqrt(max(a, 0.0) * max(b, 0.0))})
gig_df = pd.DataFrame(gig_rows).set_index('Distribution')
print("Subordinator parameters in the GIG embedding (p, a, b):\n")
print(gig_df.round(6).to_string())


### 4.1 Orbit invariants — same distribution check

Under $Y \to s\,Y$ the parameters transform but the distribution is
unchanged. The **invariants** below should agree across regularizations
(within a family) and — when the EM fits land at the same maximum —
between GH and FactorGH for matched data:

- $p$ — GIG order
- $\|\mu\|_2$ — location magnitude
- $a \cdot b$ — GIG concentration product
- $\|\gamma \cdot E[Y]\|_2$ — orbit-invariant skew vector
- $\log|\mathrm{Cov}[X]|$ — log-determinant of the marginal covariance

If two fits disagree on any of these, they are **genuinely different
distributions** (e.g. different EM modes), not the same fit on different
orbits.


In [ ]:
def _ab(model, name):
    p, a, b = gig_pab(model, name)
    return float(a * b)


def _gamma_E_Y(model):
    """Orbit-invariant skew vector γ · E[Y]."""
    sub = (model.subordinator if hasattr(model, 'subordinator')
           and not callable(model.subordinator)
           else model._joint.subordinator())
    if callable(sub):
        sub = sub()
    return np.asarray(model.gamma) * float(sub.mean())


def _logdet_cov(model):
    sign, ld = np.linalg.slogdet(np.asarray(model.cov()))
    return float(ld)


inv_rows = []
for name in DIST_ORDER:
    m = fitted[name]
    p, a, b = gig_pab(m, name)
    inv_rows.append({
        'Distribution': name,
        'p':              p,
        '||μ||':          float(np.linalg.norm(m.mu)),
        'a·b':            float(a * b),
        '||γ·E[Y]||':     float(np.linalg.norm(_gamma_E_Y(m))),
        'log|Cov[X]|':    _logdet_cov(m),
    })
inv_df = pd.DataFrame(inv_rows).set_index('Distribution')
print("Orbit-invariant summaries:\n")
print(inv_df.round(6).to_string())


In [ ]:
# μ and γ heatmaps over the magnificent 7 across all distributions.
mu_matrix = np.stack([get_mu(fitted[n])[mag7_idx] for n in DIST_ORDER])
gamma_matrix = np.stack([get_gamma(fitted[n])[mag7_idx] for n in DIST_ORDER])

fig, axes = plt.subplots(1, 2, figsize=(14, 4 + 0.3 * len(DIST_ORDER)))

for ax, mat, title in [
    (axes[0], mu_matrix, r'Location $\mu$  (×$10^{-3}$)'),
    (axes[1], gamma_matrix, r'Skewness $\gamma$  (×$10^{-3}$)'),
]:
    vmax = float(np.max(np.abs(mat))) * 1e3
    im = ax.imshow(mat * 1e3, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                   aspect='auto')
    ax.set_xticks(range(len(MAG7)))
    ax.set_xticklabels(MAG7)
    ax.set_yticks(range(len(DIST_ORDER)))
    ax.set_yticklabels(DIST_ORDER)
    ax.set_title(title)
    for (i, j), v in np.ndenumerate(mat * 1e3):
        ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                fontsize=8, color='black' if abs(v) < 0.7 * vmax else 'white')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


In [ ]:
# Σ submatrix on the Magnificent 7, one heatmap per model.
n_models = len(DIST_ORDER)
n_cols = 4
n_rows = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.5 * n_rows))
axes = np.atleast_2d(axes).flatten()

# Use a shared colour scale per family so heatmaps are comparable.
sigma_subs = {n: get_sigma(fitted[n])[np.ix_(mag7_idx, mag7_idx)]
              for n in DIST_ORDER}
vmax = max(float(np.max(np.abs(s))) for s in sigma_subs.values())

for k, name in enumerate(DIST_ORDER):
    ax = axes[k]
    s = sigma_subs[name] * 1e4   # display in 1e-4 units (≈ daily variance)
    im = ax.imshow(s, cmap='RdBu_r', vmin=-vmax * 1e4, vmax=vmax * 1e4)
    ax.set_xticks(range(len(MAG7)))
    ax.set_xticklabels(MAG7, rotation=45, fontsize=8)
    ax.set_yticks(range(len(MAG7)))
    ax.set_yticklabels(MAG7, fontsize=8)
    ax.set_title(name)
    for (i, j), v in np.ndenumerate(s):
        ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=7,
                color='black' if abs(v) < 0.7 * vmax * 1e4 else 'white')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

for k in range(len(DIST_ORDER), len(axes)):
    axes[k].set_visible(False)

plt.suptitle(r'$\Sigma$ submatrix on the Magnificent 7  (×$10^{-4}$)',
             y=1.0, fontsize=13)
plt.tight_layout()
plt.show()


## 5. Train / Test Log-Likelihood


In [ ]:
ll_rows = []
for name in DIST_ORDER:
    m = fitted[name]
    ll_rows.append({
        'Distribution': name,
        'Train LL': float(m.marginal_log_likelihood(X_train)),
        'Test LL':  float(m.marginal_log_likelihood(X_test)),
        'Fit time (s)': fit_times[name],
        'EM iters': int(em_results[name].n_iter),
    })
ll_df = pd.DataFrame(ll_rows).set_index('Distribution')
ll_df = ll_df.sort_values('Test LL', ascending=False)
print(ll_df.round(4).to_string())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

names = list(ll_df.index)
colors = plt.cm.tab10(np.arange(len(names)) % 10)

axes[0].bar(names, ll_df['Train LL'].values, color=colors)
axes[0].set_title('Mean log-likelihood — train')
axes[0].set_ylabel(r'$\langle \log f(x) \rangle$')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(names, ll_df['Test LL'].values, color=colors)
axes[1].set_title('Mean log-likelihood — test (out-of-sample)')
axes[1].set_ylabel(r'$\langle \log f(x) \rangle$')
axes[1].tick_params(axis='x', rotation=45)

axes[2].bar(names, ll_df['Fit time (s)'].values, color=colors)
axes[2].set_title('EM wall-clock time')
axes[2].set_ylabel('seconds')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


## 6. Spectrum of $\Sigma$ and Condition Number

The eigenvalues of $\Sigma$ in descending order summarise the conditioning
of each fitted dispersion. Factor models cap the rank of the
factor-loading block at $r = 50$, so their spectrum is "rank-$r$ plus
diagonal floor" rather than the gradual decay of a full covariance.


In [ ]:
eig_data = {}
cond_rows = []
for name in DIST_ORDER:
    Sigma = get_sigma(fitted[name])
    eigvals = np.linalg.eigvalsh(0.5 * (Sigma + Sigma.T))
    eigvals = np.sort(eigvals)[::-1]
    eig_data[name] = eigvals
    cond_rows.append({
        'Distribution': name,
        'λ_max': float(eigvals[0]),
        'λ_min': float(eigvals[-1]),
        'log10 cond(Σ)': float(np.log10(eigvals[0] / max(eigvals[-1], 1e-30))),
    })
cond_df = pd.DataFrame(cond_rows).set_index('Distribution')
print("Σ spectrum summary (log10 condition number):\n")
print(cond_df.round(4).to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale — top eigenvalues only.
ax = axes[0]
top_k = min(80, d)
for name, eigvals in eig_data.items():
    style = '-' if not name.startswith('Factor') else '--'
    ax.plot(np.arange(1, top_k + 1), eigvals[:top_k],
            style, label=name, lw=1.5)
ax.set_xlabel('Eigenvalue index (sorted descending)')
ax.set_ylabel(r'$\lambda_k(\Sigma)$')
ax.set_title(f'Top {top_k} eigenvalues — linear scale')
ax.legend(fontsize=9, ncol=2)

# Log scale — full spectrum.
ax = axes[1]
for name, eigvals in eig_data.items():
    style = '-' if not name.startswith('Factor') else '--'
    ax.semilogy(np.arange(1, d + 1), np.maximum(eigvals, 1e-30),
                style, label=name, lw=1.2)
ax.axvline(R_FACTORS, color='gray', linestyle=':', alpha=0.6,
           label=f'r = {R_FACTORS}')
ax.set_xlabel('Eigenvalue index')
ax.set_ylabel(r'$\lambda_k(\Sigma)$  (log scale)')
ax.set_title('Full spectrum — log scale')
ax.legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.show()


## 7. Random-Portfolio Tests

For each fitted model we sample a synthetic returns matrix and project
both the held-out test set and the synthetic samples onto a common bank
of random Dirichlet portfolio weights. Per portfolio we compute the
Kolmogorov–Smirnov statistic and the Anderson–Darling 2-sample statistic
between the empirical (test) and theoretical (sample) projections.
Smaller values indicate a closer fit.


In [ ]:
def generate_random_weights(d, n_portfolios=100, random_state=None):
    rng = np.random.default_rng(random_state)
    return rng.dirichlet(np.ones(d), size=n_portfolios)


def portfolio_test(dist, returns_test, weights, n_samples=5000, seed=42):
    samples = np.asarray(dist.rvs(n_samples, seed=seed))
    n_p = weights.shape[0]
    ks_s = np.empty(n_p); ks_p = np.empty(n_p); ad_s = np.full(n_p, np.nan)
    for i in range(n_p):
        w = weights[i]
        emp = returns_test @ w
        the = samples @ w
        ks_s[i], ks_p[i] = ks_2samp(emp, the)
        try:
            ad_s[i] = anderson_ksamp([emp, the]).statistic
        except Exception:
            pass
    return {
        'ks_stats': ks_s, 'ks_pvals': ks_p, 'ad_stats': ad_s,
        'ks_mean': ks_s.mean(),  'ks_median': np.median(ks_s),
        'ad_mean': np.nanmean(ad_s), 'ad_median': np.nanmedian(ad_s),
        'ks_reject_rate': float(np.mean(ks_p < 0.05)),
    }


N_PORTFOLIOS = 100
random_weights = generate_random_weights(d, N_PORTFOLIOS, random_state=42)
print(f"Generated {N_PORTFOLIOS} Dirichlet random portfolios over d={d} stocks.")


In [ ]:
portfolio_results = {}
for name in DIST_ORDER:
    print(f"  testing {name}...", end=' ', flush=True)
    portfolio_results[name] = portfolio_test(
        fitted[name], returns_test, random_weights,
        n_samples=5000, seed=42)
    r = portfolio_results[name]
    print(f"KS={r['ks_mean']:.4f} (reject {r['ks_reject_rate']:.0%})  "
          f"AD={r['ad_mean']:.3f}")


In [ ]:
test_rows = []
for name, r in portfolio_results.items():
    test_rows.append({
        'Distribution': name,
        'KS Mean': r['ks_mean'], 'KS Median': r['ks_median'],
        'KS Reject %': 100 * r['ks_reject_rate'],
        'AD Mean': r['ad_mean'], 'AD Median': r['ad_median'],
    })
test_df = pd.DataFrame(test_rows).set_index('Distribution').sort_values('KS Mean')
print(test_df.round(4).to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ks_data = [portfolio_results[n]['ks_stats'] for n in DIST_ORDER]
ax = axes[0]
bp = ax.boxplot(ks_data, labels=DIST_ORDER, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax.set_ylabel('KS statistic')
ax.set_title('KS statistics across random portfolios (lower = better)')
ax.tick_params(axis='x', rotation=45)

ad_data = [portfolio_results[n]['ad_stats'] for n in DIST_ORDER]
ax = axes[1]
bp = ax.boxplot(ad_data, labels=DIST_ORDER, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax.set_ylabel('AD statistic')
ax.set_title('Anderson–Darling statistics across random portfolios')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


## 8. Quantile Comparison

For every random portfolio we compare empirical (held-out) vs theoretical
(model-sampled) quantiles. The plotted band gives the 5–95% range across
portfolios; a perfectly calibrated model lies on the diagonal.


In [ ]:
def compute_quantile_comparison(dist, returns_test, weights,
                                quantiles=np.linspace(0.01, 0.99, 99),
                                n_samples=10000, seed=42):
    samples = np.asarray(dist.rvs(n_samples, seed=seed))
    n_p = weights.shape[0]
    n_q = len(quantiles)
    emp = np.empty((n_p, n_q))
    the = np.empty((n_p, n_q))
    for i in range(n_p):
        w = weights[i]
        emp[i] = np.quantile(returns_test @ w, quantiles)
        the[i] = np.quantile(samples @ w, quantiles)
    return emp, the, quantiles


quantile_results = {}
quantiles = np.linspace(0.01, 0.99, 99)
for name in DIST_ORDER:
    print(f"  quantiles {name}...")
    emp, the, q = compute_quantile_comparison(
        fitted[name], returns_test, random_weights,
        quantiles=quantiles, n_samples=10000, seed=42)
    quantile_results[name] = (emp, the, q)


In [ ]:
n_models = len(DIST_ORDER)
n_cols = 4
n_rows = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.2 * n_cols, 3.6 * n_rows))
axes = np.atleast_2d(axes).flatten()

for k, name in enumerate(DIST_ORDER):
    emp, the, q = quantile_results[name]
    emp_med = np.median(emp, axis=0)
    the_med = np.median(the, axis=0)
    the_lo  = np.percentile(the, 5, axis=0)
    the_hi  = np.percentile(the, 95, axis=0)

    ax = axes[k]
    lo = min(emp_med.min(), the_med.min())
    hi = max(emp_med.max(), the_med.max())
    ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.5)
    ax.plot(the_med, emp_med, '-', lw=2, label='median')
    ax.fill_betweenx(emp_med, the_lo, the_hi, alpha=0.3,
                     label='5–95% band')
    ax.set_xlabel('Theoretical quantile')
    ax.set_ylabel('Empirical quantile')
    ax.set_title(name)
    ax.legend(fontsize=8, loc='upper left')

for k in range(len(DIST_ORDER), len(axes)):
    axes[k].set_visible(False)

plt.suptitle(
    'QQ plots — empirical (test) vs theoretical (model-sampled), '
    f'{N_PORTFOLIOS} random portfolios',
    y=1.0, fontsize=13)
plt.tight_layout()
plt.show()


## Summary

- The factor family stores $\Sigma$ as $F F^\top + \mathrm{diag}(D)$ with
  $r = 50$ latent factors. With $d \approx 470$ stocks and only ~1300
  training days, the full-covariance fits are borderline rank-deficient
  (see the spectrum plot); the factor variant explicitly truncates to a
  well-conditioned rank-$r$ subspace.
- The GIG triple $(p, a, b)$ summarises every subordinator: VG and NInvG
  collapse to the boundary ($b = 0$ or $a = 0$ respectively), NIG fixes
  $p = -\tfrac{1}{2}$, and only GH explores the interior — see the table
  in §4.
- Compare train vs test log-likelihoods to detect over-fitting; compare
  the KS / AD distributions in §7 and the QQ bands in §8 for a
  distributional diagnostic of marginal fit on random projections.
